In [ ]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>

// Neceessary library
#include <cuda_runtime.h>

// --- CUDA Kernel Naive ---
// Global function: callable from Host (CPU), executed on Device (GPU)
__global__ void matMulNaive(const float * a, const float * b, float * c, int n) {

    // Thread mapping: calculating global 2D coordinates for the output matrix C
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Boundary check
    if (row < n && col < n) {
        float sum = 0.0f;
        // Matrix multiplication: dot product by linearizing 2D indices to access 1D flat arrays
        for (int k = 0; k < n; ++k) {
            sum += a[row * n + k] * b[k * n + col];
        }
        // Output
        c[row * n + col] = sum;
    }
}


int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);


    // Host Memory Allocation (1D Flattening)
    size_t bytes = (size_t)n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy data from host to device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel Launch: grid and block dimensions
    // - 1024 threads per Block
    // - Total numbers of block necessary to compute the whole C matrix n x n
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    printf("Starting the computation...\\n");

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch of the kernel + timing
    cudaEventRecord(start);
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaEventRecord(stop);

    // Wait for the kernel to finish
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Execution Time: %.4f seconds\\n", milliseconds / 1000.0f);

    // Copy data from device to host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample_size = (n < 1000) ? n : 1000;
        for (int i = 0; i < sample_size; i++) {
            for (int j = 0; j < sample_size; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    // Memory Deallocation: both Host and Device
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_naive.cu', 'w') as f:
    f.write(cell_str)

In [ ]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_naive.cu -o cuda_matrixmult_naive
!nvprof ./cuda_matrixmult_naive 20000

==7290== NVPROF is profiling process 7290, command: ./cuda_matrixmult_naive 20000
Starting the computation...\nExecution Time: 36.6180 seconds\n==7290== Profiling application: ./cuda_matrixmult_naive 20000
==7290== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.19%  36.6172s         1  36.6172s  36.6172s  36.6172s  matMulNaive(float const *, float const *, float*, int)
                    1.86%  699.60ms         2  349.80ms  347.89ms  351.72ms  [CUDA memcpy HtoD]
                    0.95%  359.61ms         1  359.61ms  359.61ms  359.61ms  [CUDA memcpy DtoH]
      API calls:   96.69%  36.6172s         1  36.6172s  36.6172s  36.6172s  cudaEventSynchronize
                    2.80%  1.06006s         3  353.35ms  348.11ms  360.04ms  cudaMemcpy
                    0.48%  182.86ms         3  60.952ms  143.14us  182.56ms  cudaMalloc
                    0.02%  6.2108ms         3  2.0703ms  1.3756ms  2.7731ms  cudaFree
 